# Data Augmentation Notebook

This notebook performs data augmentation on the training data from the `raw` folder and saves the augmented data to the `augmented` folder.

## Augmentation Techniques:
- Rotation
- Zooming (Scaling)
- Sheering
- Translation
- Deformation grid (elastic deformation)


In [ ]:
# Setup and Imports
import sys
from pathlib import Path
import pickle
import gzip
import numpy as np
from tqdm.notebook import tqdm
import random
from scipy.ndimage import rotate
import cv2

import elasticdeform

# PyTorch and GPU acceleration
import torch
import kornia.augmentation as K
import kornia.geometry.transform as KT

# Path setup
current_dir = Path.cwd()
BASE_PATH = current_dir.parent

RAW_DATA_PATH = BASE_PATH / 'data' / 'raw'
AUGMENTED_DATA_PATH = BASE_PATH / 'data' / 'augmented'

# Create augmented directory if it doesn't exist
AUGMENTED_DATA_PATH.mkdir(parents=True, exist_ok=True)

# GPU setup
USE_GPU = torch.cuda.is_available()
DEVICE = torch.device('cuda' if USE_GPU else 'cpu')
print(f"Using device: {DEVICE}")
if USE_GPU:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

print(f"Base path: {BASE_PATH}")
print(f"Raw data path: {RAW_DATA_PATH}")
print(f"Augmented data path: {AUGMENTED_DATA_PATH}")


## Augmentation Functions

Each function applies a specific transformation to both the image and mask to keep them aligned.


In [ ]:
def apply_rotation(image, mask, angle_range=(-15, 15)):
    """
    Apply rotation augmentation to image and mask.
    
    Args:
        image: numpy array of shape (H, W)
        mask: numpy array of shape (H, W), boolean
        angle_range: tuple of (min_angle, max_angle) in degrees
    
    Returns:
        rotated_image, rotated_mask
    """
    angle = random.uniform(angle_range[0], angle_range[1])
    
    # Rotate image
    rotated_img = rotate(image, angle, axes=(0, 1), reshape=False, order=1, mode='constant', cval=0)
    rotated_img = np.clip(rotated_img, 0, 255).astype(np.uint8)
    
    # Rotate mask with nearest neighbor interpolation to preserve binary values
    rotated_mask = rotate(mask.astype(float), angle, axes=(0, 1), reshape=False, order=0, mode='constant', cval=0)
    rotated_mask = (rotated_mask > 0.5).astype(bool)
    
    return rotated_img, rotated_mask


def apply_zooming(image, mask, zoom_range=(0.9, 1.1)):
    """
    Apply zooming (scaling) augmentation to image and mask.
    
    Args:
        image: numpy array of shape (H, W)
        mask: numpy array of shape (H, W), boolean
        zoom_range: tuple of (min_zoom, max_zoom) factors
    
    Returns:
        zoomed_image, zoomed_mask
    """
    zoom_factor = random.uniform(zoom_range[0], zoom_range[1])
    H, W = image.shape
    
    # Calculate new dimensions
    new_H, new_W = int(H * zoom_factor), int(W * zoom_factor)
    
    # Resize image
    zoomed_img = cv2.resize(image, (new_W, new_H), interpolation=cv2.INTER_LINEAR)
    zoomed_mask = cv2.resize(mask.astype(np.uint8), (new_W, new_H), interpolation=cv2.INTER_NEAREST)
    
    # Crop or pad to original size
    if zoom_factor > 1.0:
        # Crop from center
        start_h = (new_H - H) // 2
        start_w = (new_W - W) // 2
        zoomed_img = zoomed_img[start_h:start_h+H, start_w:start_w+W]
        zoomed_mask = zoomed_mask[start_h:start_h+H, start_w:start_w+W]
    else:
        # Pad with zeros
        pad_h = (H - new_H) // 2
        pad_w = (W - new_W) // 2
        zoomed_img = np.pad(zoomed_img, ((pad_h, H-new_H-pad_h), (pad_w, W-new_W-pad_w)), mode='constant', constant_values=0)
        zoomed_mask = np.pad(zoomed_mask, ((pad_h, H-new_H-pad_h), (pad_w, W-new_W-pad_w)), mode='constant', constant_values=0)
    
    zoomed_mask = (zoomed_mask > 0.5).astype(bool)
    return zoomed_img, zoomed_mask

def apply_sheering(image, mask, shear_range=(-0.2, 0.2)):
    """
    Apply sheering augmentation to image and mask.
    
    Args:
        image: numpy array of shape (H, W)
        mask: numpy array of shape (H, W), boolean
        shear_range: tuple of (min_shear, max_shear) factors
    
    Returns:
        sheered_image, sheered_mask
    """
    shear = random.uniform(shear_range[0], shear_range[1])
    H, W = image.shape
    
    # Create affine transformation matrix for sheering
    # Horizontal sheering
    if random.random() > 0.5:
        # Shear along x-axis
        transform_matrix = np.array([[1, shear, 0],
                                     [0, 1, 0],
                                     [0, 0, 1]], dtype=np.float32)
    else:
        # Shear along y-axis
        transform_matrix = np.array([[1, 0, 0],
                                     [shear, 1, 0],
                                     [0, 0, 1]], dtype=np.float32)
    
    # Apply transformation
    sheered_img = cv2.warpAffine(image, transform_matrix[:2], (W, H), 
                                 flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    sheered_mask = cv2.warpAffine(mask.astype(np.uint8), transform_matrix[:2], (W, H),
                                  flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    sheered_mask = (sheered_mask > 0.5).astype(bool)
    
    return sheered_img, sheered_mask

def apply_translation(image, mask, translate_range=(-10, 10)):
    """
    Apply translation augmentation to image and mask.
    
    Args:
        image: numpy array of shape (H, W)
        mask: numpy array of shape (H, W), boolean
        translate_range: tuple of (min_translate, max_translate) in pixels
    
    Returns:
        translated_image, translated_mask
    """
    H, W = image.shape
    tx = random.randint(translate_range[0], translate_range[1])
    ty = random.randint(translate_range[0], translate_range[1])
    
    # Create translation matrix
    transform_matrix = np.array([[1, 0, tx],
                                 [0, 1, ty]], dtype=np.float32)
    
    # Apply transformation
    translated_img = cv2.warpAffine(image, transform_matrix, (W, H),
                                    flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    translated_mask = cv2.warpAffine(mask.astype(np.uint8), transform_matrix, (W, H),
                                     flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    translated_mask = (translated_mask > 0.5).astype(bool)
    
    return translated_img, translated_mask

def apply_deformation_grid(image, mask, sigma=5, points=3, alpha=50):
    """
    Apply elastic deformation using deformation grid.
    
    Args:
        image: numpy array of shape (H, W)
        mask: numpy array of shape (H, W), boolean
        sigma: standard deviation of Gaussian filter for smooth deformation
        points: number of control points for deformation grid
        alpha: scaling factor for deformation strength
    
    Returns:
        deformed_image, deformed_mask
    """
    # Stack image and mask for simultaneous deformation
    # elasticdeform expects shape (H, W, C) or (H, W)
    # We'll apply to each separately to handle different dtypes
    
    # Apply deformation to image
    deformed_img = elasticdeform.deform_random_grid(image, sigma=sigma, points=points, 
                                                     mode='constant', cval=0, order=1)
    deformed_img = np.clip(deformed_img, 0, 255).astype(np.uint8)
    
    # Apply same deformation to mask (use order=0 for nearest neighbor)
    deformed_mask = elasticdeform.deform_random_grid(mask.astype(float), sigma=sigma, points=points,
                                                      mode='constant', cval=0, order=0)
    deformed_mask = (deformed_mask > 0.5).astype(bool)
    
    return deformed_img, deformed_mask


## GPU-Accelerated Augmentation Functions

These functions use PyTorch and Kornia for GPU-accelerated transformations. They process batches of frames simultaneously for better performance.


In [ ]:
def augment_sample_gpu(sample, augmentation_prob=0.5, device='cuda', use_elastic_deform=False):
    """
    Apply random augmentations to a sample using GPU acceleration.
    Processes all frames in a batch for better performance.
    
    Args:
        sample: dict with keys 'video', 'label', 'box', 'name', 'frames', 'dataset'
        augmentation_prob: probability of applying each augmentation
        device: device to use ('cuda' or 'cpu')
        use_elastic_deform: if True, use CPU for elastic deformation (no GPU version available)
    
    Returns:
        augmented_sample: dict with same structure as input
    """
    augmented_sample = sample.copy()
    video = sample['video'].copy()  # shape: (H, W, T)
    label = sample['label'].copy()   # shape: (H, W, T)
    box = sample['box'].copy()       # shape: (H, W)
    
    H, W, T = video.shape
    
    # Decide which augmentations to apply (same for all frames)
    apply_rot = random.random() < augmentation_prob
    apply_zoom = random.random() < augmentation_prob
    apply_shear = random.random() < augmentation_prob
    apply_trans = random.random() < augmentation_prob
    apply_def = random.random() < augmentation_prob and use_elastic_deform
    
    # Convert video and label to tensors and move to device
    # Shape: (T, H, W) -> (T, 1, H, W) for batch processing
    video_tensor = torch.from_numpy(video.transpose(2, 0, 1)).float().unsqueeze(1).to(device)  # (T, 1, H, W)
    label_tensor = torch.from_numpy(label.transpose(2, 0, 1)).float().unsqueeze(1).to(device)   # (T, 1, H, W)
    
    # Apply GPU augmentations to all frames at once
    if apply_rot:
        video_tensor, label_tensor = apply_rotation_gpu(video_tensor, label_tensor, device=device)
    
    if apply_zoom:
        video_tensor, label_tensor = apply_zooming_gpu(video_tensor, label_tensor, device=device)
    
    if apply_shear:
        video_tensor, label_tensor = apply_sheering_gpu(video_tensor, label_tensor, device=device)
    
    if apply_trans:
        video_tensor, label_tensor = apply_translation_gpu(video_tensor, label_tensor, device=device)
    
    # Elastic deformation on CPU if needed (no GPU version available)
    if apply_def:
        # Move back to CPU for elastic deformation
        video_np = video_tensor.squeeze(1).cpu().numpy().transpose(1, 2, 0)  # (H, W, T)
        label_np = label_tensor.squeeze(1).cpu().numpy().transpose(1, 2, 0)  # (H, W, T)
        
        # Apply frame by frame (elasticdeform doesn't support batch)
        for t in range(T):
            frame = video_np[:, :, t]
            mask = label_np[:, :, t]
            frame, mask = apply_deformation_grid(frame, mask)
            video_np[:, :, t] = frame
            label_np[:, :, t] = mask
        
        # Convert back to tensor
        video_tensor = torch.from_numpy(video_np.transpose(2, 0, 1)).float().unsqueeze(1).to(device)
        label_tensor = torch.from_numpy(label_np.transpose(2, 0, 1)).float().unsqueeze(1).to(device)
    
    # Convert back to numpy
    video_aug = video_tensor.squeeze(1).cpu().numpy().transpose(1, 2, 0)  # (H, W, T)
    label_aug = label_tensor.squeeze(1).cpu().numpy().transpose(1, 2, 0)  # (H, W, T)
    label_aug = (label_aug > 0.5).astype(bool)
    
    # Apply same transformations to box (2D mask)
    box_tensor = torch.from_numpy(box).float().unsqueeze(0).unsqueeze(0).to(device)  # (1, 1, H, W)
    box_mask_tensor = box_tensor.clone()
    
    if apply_rot:
        box_tensor, box_mask_tensor = apply_rotation_gpu(box_tensor, box_mask_tensor, device=device)
    
    if apply_zoom:
        box_tensor, box_mask_tensor = apply_zooming_gpu(box_tensor, box_mask_tensor, device=device)
    
    if apply_shear:
        box_tensor, box_mask_tensor = apply_sheering_gpu(box_tensor, box_mask_tensor, device=device)
    
    if apply_trans:
        box_tensor, box_mask_tensor = apply_translation_gpu(box_tensor, box_mask_tensor, device=device)
    
    if apply_def:
        # Elastic deformation for box on CPU
        box_np = box_mask_tensor.squeeze().cpu().numpy()
        box_np, _ = apply_deformation_grid((box_np * 255).astype(np.uint8), box_np.astype(bool))
        box_mask_tensor = torch.from_numpy(box_np).float().unsqueeze(0).unsqueeze(0).to(device)
    
    box_aug = (box_mask_tensor.squeeze().cpu().numpy() > 0.5).astype(bool)
    
    augmented_sample['video'] = video_aug.astype(np.uint8)
    augmented_sample['label'] = label_aug
    augmented_sample['box'] = box_aug
    
    return augmented_sample

In [ ]:
def apply_rotation_gpu(image_tensor, mask_tensor, angle_range=(-15, 15), device='cuda'):
    """
    Apply rotation augmentation to image and mask tensors on GPU.
    
    Args:
        image_tensor: torch tensor of shape (B, 1, H, W) or (B, H, W)
        mask_tensor: torch tensor of shape (B, 1, H, W) or (B, H, W)
        angle_range: tuple of (min_angle, max_angle) in degrees
        device: device to use ('cuda' or 'cpu')
    
    Returns:
        rotated_image_tensor, rotated_mask_tensor
    """
    # Ensure 4D tensors (B, C, H, W)
    if image_tensor.dim() == 3:
        image_tensor = image_tensor.unsqueeze(1)
    if mask_tensor.dim() == 3:
        mask_tensor = mask_tensor.unsqueeze(1)
    
    B, C, H, W = image_tensor.shape
    angle = random.uniform(angle_range[0], angle_range[1])
    
    # Create rotation transform matrix
    center = torch.tensor([[W/2, H/2]], device=device, dtype=torch.float32).expand(B, -1)
    angle_rad = torch.tensor([angle], device=device, dtype=torch.float32).expand(B)
    scale = torch.ones(B, device=device, dtype=torch.float32)
    
    # Get rotation matrix
    M = KT.get_rotation_matrix2d(center, angle_rad, scale)
    
    # Apply rotation with bilinear interpolation for image
    rotated_img = KT.warp_affine(image_tensor, M, dsize=(H, W), mode='bilinear', padding_mode='zeros', align_corners=False)
    rotated_img = torch.clamp(rotated_img, 0, 255)
    
    # Apply rotation with nearest neighbor for mask (preserve binary values)
    rotated_mask = KT.warp_affine(mask_tensor.float(), M, dsize=(H, W), mode='nearest', padding_mode='zeros', align_corners=False)
    rotated_mask = (rotated_mask > 0.5).float()
    
    return rotated_img, rotated_mask


def apply_zooming_gpu(image_tensor, mask_tensor, zoom_range=(0.9, 1.1), device='cuda'):
    """
    Apply zooming (scaling) augmentation to image and mask tensors on GPU.
    
    Args:
        image_tensor: torch tensor of shape (B, 1, H, W) or (B, H, W)
        mask_tensor: torch tensor of shape (B, 1, H, W) or (B, H, W)
        zoom_range: tuple of (min_zoom, max_zoom) factors
        device: device to use ('cuda' or 'cpu')
    
    Returns:
        zoomed_image_tensor, zoomed_mask_tensor
    """
    # Ensure 4D tensors
    if image_tensor.dim() == 3:
        image_tensor = image_tensor.unsqueeze(1)
    if mask_tensor.dim() == 3:
        mask_tensor = mask_tensor.unsqueeze(1)
    
    B, C, H, W = image_tensor.shape
    zoom_factor = random.uniform(zoom_range[0], zoom_range[1])
    
    # Create scale transform matrix
    center = torch.tensor([[W/2, H/2]], device=device, dtype=torch.float32).expand(B, -1)
    angle = torch.zeros(B, device=device, dtype=torch.float32)
    scale = torch.tensor([zoom_factor], device=device, dtype=torch.float32).expand(B)
    
    M = KT.get_rotation_matrix2d(center, angle, scale)
    
    # Apply scaling
    zoomed_img = KT.warp_affine(image_tensor, M, dsize=(H, W), mode='bilinear', padding_mode='zeros', align_corners=False)
    zoomed_img = torch.clamp(zoomed_img, 0, 255)
    
    zoomed_mask = KT.warp_affine(mask_tensor.float(), M, dsize=(H, W), mode='nearest', padding_mode='zeros', align_corners=False)
    zoomed_mask = (zoomed_mask > 0.5).float()
    
    return zoomed_img, zoomed_mask


def apply_sheering_gpu(image_tensor, mask_tensor, shear_range=(-0.2, 0.2), device='cuda'):
    """
    Apply sheering augmentation to image and mask tensors on GPU.
    
    Args:
        image_tensor: torch tensor of shape (B, 1, H, W) or (B, H, W)
        mask_tensor: torch tensor of shape (B, 1, H, W) or (B, H, W)
        shear_range: tuple of (min_shear, max_shear) factors
        device: device to use ('cuda' or 'cpu')
    
    Returns:
        sheered_image_tensor, sheered_mask_tensor
    """
    # Ensure 4D tensors
    if image_tensor.dim() == 3:
        image_tensor = image_tensor.unsqueeze(1)
    if mask_tensor.dim() == 3:
        mask_tensor = mask_tensor.unsqueeze(1)
    
    B, C, H, W = image_tensor.shape
    shear = random.uniform(shear_range[0], shear_range[1])
    
    # Create shear matrix
    if random.random() > 0.5:
        # Shear along x-axis
        M = torch.tensor([[1.0, shear, 0.0],
                          [0.0, 1.0, 0.0]], device=device, dtype=torch.float32)
    else:
        # Shear along y-axis
        M = torch.tensor([[1.0, 0.0, 0.0],
                          [shear, 1.0, 0.0]], device=device, dtype=torch.float32)
    
    M = M.unsqueeze(0).expand(B, -1, -1)
    
    # Apply shear transformation
    sheered_img = KT.warp_affine(image_tensor, M, dsize=(H, W), mode='bilinear', padding_mode='zeros', align_corners=False)
    sheered_img = torch.clamp(sheered_img, 0, 255)
    
    sheered_mask = KT.warp_affine(mask_tensor.float(), M, dsize=(H, W), mode='nearest', padding_mode='zeros', align_corners=False)
    sheered_mask = (sheered_mask > 0.5).float()
    
    return sheered_img, sheered_mask


def apply_translation_gpu(image_tensor, mask_tensor, translate_range=(-10, 10), device='cuda'):
    """
    Apply translation augmentation to image and mask tensors on GPU.
    
    Args:
        image_tensor: torch tensor of shape (B, 1, H, W) or (B, H, W)
        mask_tensor: torch tensor of shape (B, 1, H, W) or (B, H, W)
        translate_range: tuple of (min_translate, max_translate) in pixels
        device: device to use ('cuda' or 'cpu')
    
    Returns:
        translated_image_tensor, translated_mask_tensor
    """
    # Ensure 4D tensors
    if image_tensor.dim() == 3:
        image_tensor = image_tensor.unsqueeze(1)
    if mask_tensor.dim() == 3:
        mask_tensor = mask_tensor.unsqueeze(1)
    
    B, C, H, W = image_tensor.shape
    tx = random.randint(translate_range[0], translate_range[1])
    ty = random.randint(translate_range[0], translate_range[1])
    
    # Create translation matrix
    M = torch.tensor([[1.0, 0.0, float(tx)],
                      [0.0, 1.0, float(ty)]], device=device, dtype=torch.float32)
    M = M.unsqueeze(0).expand(B, -1, -1)
    
    # Apply translation
    translated_img = KT.warp_affine(image_tensor, M, dsize=(H, W), mode='bilinear', padding_mode='zeros', align_corners=False)
    translated_img = torch.clamp(translated_img, 0, 255)
    
    translated_mask = KT.warp_affine(mask_tensor.float(), M, dsize=(H, W), mode='nearest', padding_mode='zeros', align_corners=False)
    translated_mask = (translated_mask > 0.5).float()
    
    return translated_img, translated_mask


## Main Augmentation Function

This function applies random augmentations to a single sample (video + labels).


In [ ]:
def augment_sample(sample, augmentation_prob=0.5):
    """
    Apply random augmentations to a sample.
    
    Args:
        sample: dict with keys 'video', 'label', 'box', 'name', 'frames', 'dataset'
        augmentation_prob: probability of applying each augmentation
    
    Returns:
        augmented_sample: dict with same structure as input
    """
    augmented_sample = sample.copy()
    video = sample['video'].copy()  # shape: (H, W, T)
    label = sample['label'].copy()   # shape: (H, W, T)
    box = sample['box'].copy()       # shape: (H, W)
    
    H, W, T = video.shape
    
    # Apply augmentations frame by frame to keep consistency
    # We apply the same transformation to all frames in a sample
    # Choose which augmentations to apply
    apply_rot = random.random() < augmentation_prob
    apply_zoom = random.random() < augmentation_prob
    apply_shear = random.random() < augmentation_prob
    apply_trans = random.random() < augmentation_prob
    apply_def = random.random() < augmentation_prob
    
    # Process each frame
    for t in range(T):
        frame = video[:, :, t]
        mask = label[:, :, t]
        
        # Apply augmentations in sequence
        if apply_rot:
            frame, mask = apply_rotation(frame, mask)
        
        if apply_zoom:
            frame, mask = apply_zooming(frame, mask)
        
        if apply_shear:
            frame, mask = apply_sheering(frame, mask)
        
        if apply_trans:
            frame, mask = apply_translation(frame, mask)
        
        if apply_def:
            frame, mask = apply_deformation_grid(frame, mask)
        
        video[:, :, t] = frame
        label[:, :, t] = mask
    
    # Apply same transformations to box (2D mask)
    box_frame = box.copy()
    box_mask = box.copy()
    
    if apply_rot:
        box_frame, box_mask = apply_rotation(box_frame.astype(np.uint8) * 255, box_mask)
        box = box_mask
    
    if apply_zoom:
        box_frame, box_mask = apply_zooming(box_frame.astype(np.uint8), box_mask)
        box = box_mask
    
    if apply_shear:
        box_frame, box_mask = apply_sheering(box_frame.astype(np.uint8), box_mask)
        box = box_mask
    
    if apply_trans:
        box_frame, box_mask = apply_translation(box_frame.astype(np.uint8), box_mask)
        box = box_mask
    
    if apply_def:
        box_frame, box_mask = apply_deformation_grid(box_frame.astype(np.uint8), box_mask)
        box = box_mask
    
    augmented_sample['video'] = video
    augmented_sample['label'] = label
    augmented_sample['box'] = box
    
    return augmented_sample


## Helper Functions

Functions for loading and saving gzipped pickle files.


In [ ]:
def load_zipped_pickle(filename):
    """Load a gzipped pickle file."""
    with gzip.open(filename, 'rb') as f:
        loaded_object = pickle.load(f)
        return loaded_object

def save_zipped_pickle(obj, filename):
    """Save an object to a gzipped pickle file."""
    with gzip.open(filename, 'wb') as f:
        pickle.dump(obj, f, 2)


In [ ]:
# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

# Load training data
print("Loading training data...")
train_data = load_zipped_pickle(str(RAW_DATA_PATH / 'train.pkl'))
print(f"Loaded {len(train_data)} training samples")

# Load test data (optional, for reference)
print("Loading test data...")
test_data = load_zipped_pickle(str(RAW_DATA_PATH / 'test.pkl'))
print(f"Loaded {len(test_data)} test samples")


In [ ]:
# Configuration
NUM_AUGMENTATIONS = 2  # Number of augmented versions per original sample
AUGMENTATION_PROB = 0.7  # Probability of applying each augmentation technique
USE_GPU_AUGMENTATION = USE_GPU  # Use GPU if available, otherwise fall back to CPU
USE_ELASTIC_DEFORM = False  # Set to True to enable elastic deformation (runs on CPU)

print(f"Configuration:")
print(f"  - Number of augmentations per sample: {NUM_AUGMENTATIONS}")
print(f"  - Augmentation probability: {AUGMENTATION_PROB}")
print(f"  - Use GPU acceleration: {USE_GPU_AUGMENTATION}")
print(f"  - Use elastic deformation: {USE_ELASTIC_DEFORM}")
print(f"  - Total samples to create: {len(train_data) * (1 + NUM_AUGMENTATIONS)}")


In [ ]:
# Create augmented dataset
augmented_train_data = []

# Start with original data
print("Adding original training samples...")
for sample in tqdm(train_data, desc="Original samples"):
    augmented_train_data.append(sample.copy())

# Add augmented samples
print(f"Creating {NUM_AUGMENTATIONS} augmented versions per sample...")
print(f"Using {'GPU' if USE_GPU_AUGMENTATION else 'CPU'} acceleration...")

if USE_GPU_AUGMENTATION:
    # Use GPU-accelerated augmentation
    for sample in tqdm(train_data, desc="Augmenting samples (GPU)"):
        for aug_idx in range(NUM_AUGMENTATIONS):
            augmented_sample = augment_sample_gpu(
                sample, 
                augmentation_prob=AUGMENTATION_PROB,
                device=DEVICE,
                use_elastic_deform=USE_ELASTIC_DEFORM
            )
            # Update name to indicate augmentation
            augmented_sample['name'] = f"{sample['name']}_aug{aug_idx+1}"
            augmented_train_data.append(augmented_sample)
else:
    # Use CPU augmentation
    for sample in tqdm(train_data, desc="Augmenting samples (CPU)"):
        for aug_idx in range(NUM_AUGMENTATIONS):
            augmented_sample = augment_sample(sample, augmentation_prob=AUGMENTATION_PROB)
            # Update name to indicate augmentation
            augmented_sample['name'] = f"{sample['name']}_aug{aug_idx+1}"
            augmented_train_data.append(augmented_sample)

print(f"\nTotal augmented training samples: {len(augmented_train_data)}")
print(f"  - Original: {len(train_data)}")
print(f"  - Augmented: {len(augmented_train_data) - len(train_data)}")


In [ ]:
# Save augmented training data
output_path = AUGMENTED_DATA_PATH / 'train_augmented.pkl'
print(f"Saving augmented training data to {output_path}...")
save_zipped_pickle(augmented_train_data, str(output_path))
print("Done!")

# Also save test data (unchanged) for convenience
test_output_path = AUGMENTED_DATA_PATH / 'test.pkl'
print(f"Saving test data to {test_output_path}...")
save_zipped_pickle(test_data, str(test_output_path))
print("Done!")


In [ ]:
# Verify saved data
print("Verifying saved augmented data...")
loaded_augmented = load_zipped_pickle(str(AUGMENTED_DATA_PATH / 'train_augmented.pkl'))
print(f"Loaded {len(loaded_augmented)} samples")

# Inspect a few samples
print("\nSample inspection:")
for i, sample in enumerate(loaded_augmented[:3]):
    print(f"\nSample {i}:")
    print(f"  Name: {sample.get('name')}")
    vid = sample.get('video')
    lab = sample.get('label')
    box = sample.get('box')
    print(f"  Video: shape={vid.shape}, dtype={vid.dtype}")
    print(f"  Label: shape={lab.shape}, dtype={lab.dtype}")
    print(f"  Box: shape={box.shape}, dtype={box.dtype}")
    print(f"  Frames: {sample.get('frames')}")
    print(f"  Dataset: {sample.get('dataset')}")


In [ ]:
# Visualize original vs augmented sample
import matplotlib.pyplot as plt

# Find an original sample and its augmented version
original_sample = train_data[0]
augmented_sample = None
for sample in loaded_augmented:
    if sample['name'].startswith(original_sample['name'] + '_aug'):
        augmented_sample = sample
        break

if augmented_sample is not None:
    # Get a frame from the middle
    frame_idx = original_sample['frames'][len(original_sample['frames'])//2]
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Original
    axes[0, 0].imshow(original_sample['video'][:, :, frame_idx], cmap='gray')
    axes[0, 0].set_title(f"Original Video - Frame {frame_idx}")
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(original_sample['label'][:, :, frame_idx], cmap='gray')
    axes[0, 1].set_title("Original Label")
    axes[0, 1].axis('off')
    
    # Overlay
    overlay_orig = original_sample['video'][:, :, frame_idx].copy()
    overlay_orig = np.stack([overlay_orig, overlay_orig, overlay_orig], axis=-1)
    mask_orig = original_sample['label'][:, :, frame_idx]
    overlay_orig[:, :, 1] = np.where(mask_orig, overlay_orig[:, :, 1] * 0.5 + 255 * 0.5, overlay_orig[:, :, 1])
    axes[0, 2].imshow(overlay_orig.astype(np.uint8))
    axes[0, 2].set_title("Original Overlay")
    axes[0, 2].axis('off')
    
    # Augmented
    aug_frame_idx = augmented_sample['frames'][len(augmented_sample['frames'])//2] if len(augmented_sample['frames']) > 0 else frame_idx
    axes[1, 0].imshow(augmented_sample['video'][:, :, aug_frame_idx], cmap='gray')
    axes[1, 0].set_title(f"Augmented Video - Frame {aug_frame_idx}")
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(augmented_sample['label'][:, :, aug_frame_idx], cmap='gray')
    axes[1, 1].set_title("Augmented Label")
    axes[1, 1].axis('off')
    
    # Overlay
    overlay_aug = augmented_sample['video'][:, :, aug_frame_idx].copy()
    overlay_aug = np.stack([overlay_aug, overlay_aug, overlay_aug], axis=-1)
    mask_aug = augmented_sample['label'][:, :, aug_frame_idx]
    overlay_aug[:, :, 1] = np.where(mask_aug, overlay_aug[:, :, 1] * 0.5 + 255 * 0.5, overlay_aug[:, :, 1])
    axes[1, 2].imshow(overlay_aug.astype(np.uint8))
    axes[1, 2].set_title("Augmented Overlay")
    axes[1, 2].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("Could not find matching augmented sample for visualization")
